# Vizualizacie novych experimentov

Notebook generuje prehladove grafy a sumarne CSV pre:
- event experimenty (`2_modelovanie/Eventy/predictions`)
- kontinualne experimenty (defaultne `DST+BZ_GSM+V`)

Casove okna su volitelne. Ak `start_date` a `end_date` ostanu `None`, vytvoria sa len sumarne grafy.

In [1]:
from pathlib import Path

from viz_nove_experimenty import (
    OUTPUT_ROOT,
    available_continuous_horizons,
    available_event_lookbacks,
    plot_continuous_lines_by_lookback,
    plot_continuous_metric_heatmap,
    plot_continuous_vs_event_window,
    plot_continuous_window_grid,
    plot_event_lines_by_feature,
    plot_event_metric_heatmaps,
    plot_event_window_by_feature,
    save_continuous_summary_csv,
    save_event_summary_csv,
    summarize_continuous_predictions,
    summarize_event_predictions,
)


In [2]:
# Konfiguracia
continuous_variant = "DST+BZ_GSM+V"
event_feature_key = "dst_bz_v"

# Nastav len ak chces vytvarat aj casove grafy
start_date = None
end_date = None

# Pouzije sa len pre casove grafy
window_horizon = 1
window_lookback = 6

output_dir = OUTPUT_ROOT
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

PosixPath('/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/3_vizualizacie/vystupy_nove_data')

In [3]:
continuous_summary = summarize_continuous_predictions(continuous_variant)
event_summary = summarize_event_predictions()

continuous_csv = None
event_csv = None

if not continuous_summary.empty:
    continuous_csv = save_continuous_summary_csv(continuous_summary, output_dir)
    plot_continuous_metric_heatmap(
        continuous_summary,
        metric="mae_hybrid",
        title=f"Continuous | BiLSTM+GP MAE | {continuous_variant}",
        output_path=output_dir / "continuous_mae_hybrid_heatmap.png",
    )
    plot_continuous_metric_heatmap(
        continuous_summary,
        metric="f1_m20",
        title=f"Continuous | F1 at DST <= -20 | {continuous_variant}",
        output_path=output_dir / "continuous_f1_m20_heatmap.png",
    )
    plot_continuous_lines_by_lookback(
        continuous_summary,
        output_path=output_dir / "continuous_mae_by_horizon.png",
    )

if not event_summary.empty:
    event_csv = save_event_summary_csv(event_summary, output_dir)
    plot_event_metric_heatmaps(
        event_summary,
        metric="mae_bilstm_gp",
        title_prefix="Event | BiLSTM+GP MAE",
        output_path=output_dir / "event_mae_bilstm_gp_heatmaps.png",
    )
    plot_event_metric_heatmaps(
        event_summary,
        metric="f1_m20",
        title_prefix="Event | F1 at DST <= -20",
        output_path=output_dir / "event_f1_m20_heatmaps.png",
    )
    plot_event_metric_heatmaps(
        event_summary,
        metric="f1_m50",
        title_prefix="Event | F1 at DST <= -50",
        output_path=output_dir / "event_f1_m50_heatmaps.png",
    )
    plot_event_lines_by_feature(
        event_summary,
        metric="mae_bilstm_gp",
        title="Event experiment | BiLSTM+GP MAE by horizon",
        output_path=output_dir / "event_mae_by_feature.png",
    )
    plot_event_lines_by_feature(
        event_summary,
        metric="f1_m20",
        title="Event experiment | F1 at DST <= -20 by horizon",
        output_path=output_dir / "event_f1_m20_by_feature.png",
    )

print(f"Output directory: {output_dir}")
print(f"Continuous summary rows: {0 if continuous_summary.empty else len(continuous_summary)}")
print(f"Event summary rows: {0 if event_summary.empty else len(event_summary)}")
print(f"Continuous CSV: {continuous_csv}")
print(f"Event CSV: {event_csv}")

if not continuous_summary.empty:
    display(continuous_summary.sort_values(["lookback_hours", "horizon_hours"]).head(12))
if not event_summary.empty:
    display(event_summary.sort_values(["feature_key", "lookback_hours", "horizon_hours"]).head(12))

Output directory: /home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/3_vizualizacie/vystupy_nove_data
Continuous summary rows: 24
Event summary rows: 96
Continuous CSV: /home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/3_vizualizacie/vystupy_nove_data/continuous_summary_metrics.csv
Event CSV: /home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/3_vizualizacie/vystupy_nove_data/event_summary_metrics.csv


,variant,horizon_hours,lookback_hours,rows,mae_persistence,mae_lstm,mae_hybrid,rmse_persistence,rmse_lstm,rmse_hybrid,path,precision_m20,recall_m20,f1_m20,precision_m50,recall_m50,f1_m50
3,DST+BZ_GSM+V,1,6,194193,2.567961,2.266962,2.224716,4.039853,3.619961,3.650807,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.933767,0.892037,0.912426,0.911058,0.878097,0.894274
7,DST+BZ_GSM+V,2,6,194191,4.211529,3.649597,3.768530,6.576884,5.469958,5.688656,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.870557,0.830679,0.850151,0.846668,0.790233,0.817478
11,DST+BZ_GSM+V,3,6,194189,5.299620,4.699972,4.902556,8.304162,7.054849,7.351542,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.820791,0.787264,0.803678,0.784302,0.721262,0.751462
15,DST+BZ_GSM+V,4,6,194187,6.073218,5.352043,5.600902,9.608726,8.159612,8.507945,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.786982,0.756741,0.771565,0.758871,0.670825,0.712137
19,DST+BZ_GSM+V,5,6,194185,6.683642,5.802350,5.993286,10.677962,9.090739,9.319422,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.786836,0.720443,0.752178,0.701844,0.657815,0.679117
23,DST+BZ_GSM+V,6,6,194183,7.203962,6.223081,6.442398,11.619545,9.748633,10.088440,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.781122,0.692981,0.734417,0.658982,0.611299,0.634246
0,DST+BZ_GSM+V,1,12,194187,2.567983,2.353094,2.248116,4.039893,3.417317,3.439275,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.932136,0.891959,0.911605,0.869991,0.886117,0.877980
4,DST+BZ_GSM+V,2,12,194185,4.211535,3.626103,3.742001,6.576930,5.541396,5.718136,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.870245,0.834635,0.852068,0.830334,0.805917,0.817943
8,DST+BZ_GSM+V,3,12,194183,5.299650,4.620908,4.894192,8.304239,7.010143,7.376004,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.828453,0.774869,0.800765,0.800000,0.701479,0.747507
12,DST+BZ_GSM+V,4,12,194181,6.073318,5.423658,5.568324,9.608851,8.210712,8.466751,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.811761,0.739527,0.773962,0.744373,0.660132,0.699726


,feature_key,lookback_hours,horizon_hours,rows,mae_persistence,mae_bilstm,mae_bilstm_gp,rmse_persistence,rmse_bilstm,rmse_bilstm_gp,path,precision_m20,recall_m20,f1_m20,precision_m50,recall_m50,f1_m50
18,dst_bz,6,1,22676,4.290704,5.247375,3.882114,6.806682,7.901546,6.212120,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.964050,0.965143,0.964596,0.919314,0.874918,0.896567
19,dst_bz,6,2,22676,6.995017,7.084927,6.467468,11.078006,10.832824,10.095609,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.935912,0.944560,0.940216,0.877607,0.768980,0.819710
20,dst_bz,6,3,22676,8.866334,8.397439,8.132223,14.128525,13.101051,12.750479,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.920922,0.927098,0.924000,0.831381,0.741136,0.783669
21,dst_bz,6,4,22676,10.301729,9.372087,9.573851,16.532749,14.884519,14.851106,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.894321,0.926009,0.909889,0.768990,0.693562,0.729331
22,dst_bz,6,5,22676,11.559358,10.200822,10.434634,18.566598,16.372210,16.229233,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.858367,0.954758,0.904000,0.843436,0.601957,0.702524
23,dst_bz,6,6,22676,12.739637,10.983066,11.667949,20.429605,17.736726,18.240548,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.866164,0.920524,0.892517,0.735409,0.616037,0.670451
0,dst_bz,12,1,21818,4.352186,5.343588,4.000445,6.897240,7.746711,6.307278,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.970110,0.962009,0.966043,0.926375,0.864912,0.894589
1,dst_bz,12,2,21818,7.098084,6.882305,6.474051,11.230075,10.365735,10.033544,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.945856,0.934973,0.940383,0.877155,0.785947,0.829050
2,dst_bz,12,3,21818,9.003942,8.305713,8.249901,14.329091,12.802832,12.765388,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.918152,0.930368,0.924220,0.810507,0.751795,0.780047
3,dst_bz,12,4,21818,10.468237,9.307927,9.683454,16.775116,14.694929,15.072875,/home/jovyan/data/lightning/TereziaD/DP_pokrac...,0.904977,0.924583,0.914675,0.797352,0.681166,0.734694


In [4]:
# Volitelne casove grafy
if start_date is not None or end_date is not None:
    if window_horizon in available_continuous_horizons(continuous_variant):
        plot_continuous_window_grid(
            continuous_variant=continuous_variant,
            horizon_hours=window_horizon,
            start_date=start_date,
            end_date=end_date,
            output_path=output_dir / f"continuous_window_DST+{window_horizon}.png",
        )

    if window_horizon in range(1, 7) and window_lookback in available_event_lookbacks():
        plot_event_window_by_feature(
            horizon_hours=window_horizon,
            lookback_hours=window_lookback,
            start_date=start_date,
            end_date=end_date,
            output_path=output_dir / f"event_feature_window_DST+{window_horizon}_L{window_lookback}.png",
        )

    if window_horizon in range(1, 7):
        plot_continuous_vs_event_window(
            continuous_variant=continuous_variant,
            event_feature_key=event_feature_key,
            horizon_hours=window_horizon,
            start_date=start_date,
            end_date=end_date,
            output_path=output_dir / f"continuous_vs_event_DST+{window_horizon}.png",
        )

    print("Window plots generated.")
else:
    print("Window plots skipped. Set start_date/end_date if you want them.")

Window plots skipped. Set start_date/end_date if you want them.


In [5]:
sorted(path.name for path in Path(output_dir).glob("*"))

['.ipynb_checkpoints',
 'continuous_f1_m20_heatmap.png',
 'continuous_mae_by_horizon.png',
 'continuous_mae_hybrid_heatmap.png',
 'continuous_summary_metrics.csv',
 'event_f1_m20_by_feature.png',
 'event_f1_m20_heatmaps.png',
 'event_f1_m50_heatmaps.png',
 'event_mae_bilstm_gp_heatmaps.png',
 'event_mae_by_feature.png',
 'event_summary_metrics.csv',
 'graf_DST+1']